# 10 — RQ1 đánh giá cuối: chấm điểm Hold-out, focused set và Lazada (máy local)

**Lần mở duy nhất**

Đây là nơi duy nhất đọc nhãn gold của Hold-out (1.000), focused set (150) và Lazada (500). Notebook chỉ chạy sau khi 10a (mô hình
neural, đã qua cổng Bước 0) và 10b (LLM) đã lưu đủ dự đoán. Không có bước tinh chỉnh nào; cấu hình và ngưỡng đọc từ
`03_configs/frozen/` (amendment v2.1).

**Quy ước báo cáo**

Mô hình neural được báo cáo bằng trung bình ± độ lệch chuẩn qua ba seed; seed 42 là lần chạy chuẩn (Q6) dùng cho bootstrap, ma trận
nhầm lẫn và hybrid 3×3. LLM là một lần chạy. Khoảng tin cậy 95% dùng bootstrap 10.000 lần (seed 42) với cùng các mẫu lại cho mọi
cấu hình, nên chênh lệch giữa hai cấu hình là chênh lệch ghép cặp; hai cấu hình được xem là tương đương khi CI của chênh lệch chứa 0.

**Không kết luận trong notebook**

Notebook in bảng và khoảng tin cậy; phần diễn giải được viết sau khi xem kết quả.

In [1]:
import os, sys, json, subprocess, datetime
import numpy as np, pandas as pd
ROOT = os.path.abspath('..')
sys.path.insert(0, os.path.join(ROOT, '02_src'))
from src.utils.seeds import set_seed
set_seed(42)
from src.classification import evaluate as E
from src.classification.tier2_bfu.mapping import TIER2_LABELS
from src.classification.final_eval import (LOCKED, load_frozen, neural_probs, llm_ids, pm, point_metric, bootstrap_table,
                                           hybrid_predictions)
DATA = os.path.join(ROOT, '01_data', 'processed')
RES = os.path.join(ROOT, '05_experiments', 'rq1_classification')
FZ = load_frozen(ROOT)
SEEDS = FZ['tier1']['seeds']
OPEN_DATE = datetime.date.today().strftime('%d/%m/%Y')
COMMIT = subprocess.run(['git', '-C', ROOT, 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip()
DIRTY = bool(subprocess.run(['git', '-C', ROOT, 'status', '--porcelain'], capture_output=True, text=True).stdout.strip())
print('Ngày mở Hold-out / focused / Lazada:', OPEN_DATE, '| commit:', COMMIT, '| có thay đổi chưa commit:', DIRTY)
RUN = json.load(open(os.path.join(RES, 'gd10_run_info.json'), encoding='utf-8'))
print('10a: ngày', RUN['run_date'], '| commit', RUN['commit'], '| cổng Bước 0 đạt:', RUN['gate_pass'])
assert RUN['gate_pass'], 'Cổng Bước 0 của 10a không đạt — không chấm điểm.'
NEED = [f'gd10_pred_{s}.csv' for s in LOCKED] + [f'gd10_llm_{s}_tier{t}.csv' for s in ('holdout', 'lazada') for t in (1, 2)] \
       + ['gd10_llm_focused_tier2.csv', 'gd10_costs_neural.csv', 'gd10_llm_costs.csv']
missing = [f for f in NEED if not os.path.exists(os.path.join(RES, f))]
assert not missing, f'Thiếu file: {missing}'

Ngày mở Hold-out / focused / Lazada: 26/10/2026 | commit: a79b305864bb817c31046747887655f36edc6988 | có thay đổi chưa commit: True
10a: ngày 2026-10-25 | commit a79b305864bb817c31046747887655f36edc6988 | cổng Bước 0 đạt: True


## Dữ liệu và dự đoán

**Nhãn gold**

Đọc `gold_label` của ba tập khóa, quy về id bốn lớp BUG=0, FEATURE=1, UX=2, OTHER=3.

**Dự đoán theo chiến lược**

Tầng 1 của neural lấy xác suất ACTIONABLE với ngưỡng đóng băng (ViSoBERT 0,11; Bi-LSTM 0,09); tầng 2 và baseline lấy argmax. LLM
dùng nhãn trả về; tầng 2 của LLM có giá trị −1 ở review không được gọi, và mọi chỗ cần dùng đều được kiểm tra đủ phủ.

In [2]:
FOUR = E.FOUR_CLASSES
GOLD = {s: pd.read_csv(os.path.join(DATA, f), usecols=['review_id', 'gold_label'], encoding='utf-8-sig') for s, f in LOCKED.items()}
IDS = {s: GOLD[s]['review_id'].tolist() for s in LOCKED}
Y4 = {s: GOLD[s]['gold_label'].map({l: i for i, l in enumerate(FOUR)}).to_numpy() for s in LOCKED}
PRED = {s: pd.read_csv(os.path.join(RES, f'gd10_pred_{s}.csv')) for s in LOCKED}
LLM = {}
for s, t in (('holdout', 1), ('holdout', 2), ('lazada', 1), ('lazada', 2), ('focused', 2)):
    LLM[(s, t)] = pd.read_csv(os.path.join(RES, f'gd10_llm_{s}_tier{t}.csv'))
NEURAL = {'ViSoBERT': 'visobert', 'Bi-LSTM + fastText': 'bilstm'}
STRATS = list(NEURAL) + ['LLM few-shot']

def tier1_act(s):
    out = {}
    for name, m in NEURAL.items():
        thr = FZ['tier1'][m]['threshold']
        out[name] = {sd: p[:, 1] >= thr for sd, p in neural_probs(PRED[s], IDS[s], 'tier1', m, SEEDS).items()}
    if (s, 1) in LLM:
        ids = llm_ids(IDS[s], LLM[(s, 1)])
        assert (ids >= 0).all(), f'{s}: LLM tầng 1 thiếu review'
        out['LLM few-shot'] = {'llm': ids == 1}
    return out

def tier2_ids(s):
    out = {name: {sd: p.argmax(1) for sd, p in neural_probs(PRED[s], IDS[s], 'tier2', m, SEEDS).items()}
           for name, m in NEURAL.items()}
    out['LLM few-shot'] = {'llm': llm_ids(IDS[s], LLM[(s, 2)])}
    return out

def flat_ids(s):
    return {sd: p.argmax(1) for sd, p in neural_probs(PRED[s], IDS[s], 'flat', 'visobert', SEEDS).items()}

def canon(runs):
    return runs[42] if 42 in runs else runs['llm']

for s in LOCKED:
    print(f'{s}: {len(IDS[s])} review | các lớp có mặt: {sorted(set(FOUR[i] for i in Y4[s]))}')

holdout: 1000 review | các lớp có mặt: ['BUG', 'FEATURE', 'OTHER', 'UX']
focused: 150 review | các lớp có mặt: ['BUG', 'FEATURE', 'OTHER', 'UX']
lazada: 500 review | các lớp có mặt: ['BUG', 'FEATURE', 'OTHER', 'UX']


## Tầng 1 trên Hold-out

**Chỉ số**

F1, Recall và Precision của ACTIONABLE ở ngưỡng đóng băng, Macro-F1 hai lớp; neural là trung bình ± độ lệch chuẩn qua ba seed.
Khoảng tin cậy và chênh lệch ghép cặp tính trên seed 42 và lần chạy LLM.

In [3]:
s = 'holdout'
y1 = (Y4[s] != 3).astype(int)
A1 = tier1_act(s)
rows = []
for name, runs in A1.items():
    v = {met: [point_metric(y1, a.astype(int), 2, met, None if met == 'macro_f1' else 1) for a in runs.values()]
         for met in ('f1', 'recall', 'precision', 'macro_f1')}
    rows.append({'chiến lược': name, 'F1 ACT': pm(v['f1']), 'Recall ACT': pm(v['recall']), 'Precision ACT': pm(v['precision']),
                 'Macro-F1 (2 lớp)': pm(v['macro_f1']), 'số lần chạy': len(runs)})
T1 = pd.DataFrame(rows)
print('Tầng 1 — Hold-out (n = %d; %%):' % len(y1)); print(T1.to_string(index=False))
P1 = {name: canon(runs).astype(int) for name, runs in A1.items()}
T1_CI = []
for met, cls in (('f1', 1), ('recall', 1), ('macro_f1', None)):
    ci, diff = bootstrap_table(y1, P1, 2, met, cls)
    ci.insert(0, 'chỉ số', met); diff.insert(0, 'chỉ số', met)
    T1_CI.append((ci, diff))
    print(f'\n{met} — CI 95% (seed 42 / LLM):'); print(ci.round(4).to_string(index=False)); print(diff.round(4).to_string(index=False))

Tầng 1 — Hold-out (n = 1000; %):
        chiến lược     F1 ACT Recall ACT Precision ACT Macro-F1 (2 lớp)  số lần chạy
          ViSoBERT 65.9 ± 0.9 69.8 ± 4.0    62.7 ± 2.9       78.9 ± 0.6            3
Bi-LSTM + fastText 59.7 ± 4.7 84.0 ± 7.4    47.0 ± 7.5       72.6 ± 5.0            3
      LLM few-shot       81.7       86.9          77.2             88.7            1

f1 — CI 95% (seed 42 / LLM):
chỉ số           cấu hình  giá trị  CI 2,5%  CI 97,5%
    f1           ViSoBERT   0.6492   0.5920    0.7020
    f1 Bi-LSTM + fastText   0.6217   0.5708    0.6695
    f1       LLM few-shot   0.8175   0.7737    0.8564
chỉ số                  A                  B  chênh (A − B)  CI 2,5%  CI 97,5%  CI chứa 0
    f1           ViSoBERT Bi-LSTM + fastText         0.0275  -0.0232    0.0767       True
    f1           ViSoBERT       LLM few-shot        -0.1683  -0.2260   -0.1122      False
    f1 Bi-LSTM + fastText       LLM few-shot        -0.1958  -0.2461   -0.1458      False

recall — CI 95% (see

## Tầng 2 trên gold-actionable của Hold-out

**Chỉ số**

Macro-F1 ba lớp và F1 từng lớp trên các review có nhãn gold BUG/FEATURE/UX, tách khỏi lỗi của tầng 1.

In [4]:
def tier2_table(s, title):
    mask = Y4[s] != 3
    y2 = Y4[s][mask]
    T2r = tier2_ids(s)
    rows, preds = [], {}
    for name, runs in T2r.items():
        ms = []
        for p in runs.values():
            p = p[mask]
            assert (p >= 0).all(), f'{s}/{name}: tầng 2 thiếu dự đoán trên gold-actionable'
            ms.append(E.metrics(y2, p, TIER2_LABELS))
        rows.append({'chiến lược': name, 'Macro-F1': pm([m['macro_f1'] for m in ms]),
                     **{f'F1 {c}': pm([m[f'f1_{c}'] for m in ms]) for c in TIER2_LABELS}, 'số lần chạy': len(runs)})
        preds[name] = canon(runs)[mask]
    tab = pd.DataFrame(rows)
    print(f'{title} (n = {mask.sum()}; %):'); print(tab.to_string(index=False))
    ci_m, diff_m = bootstrap_table(y2, preds, 3)
    ci_f, diff_f = bootstrap_table(y2, preds, 3, 'f1', 1)
    print('\nMacro-F1 — CI 95%:'); print(ci_m.round(4).to_string(index=False)); print(diff_m.round(4).to_string(index=False))
    print('\nF1 FEATURE — CI 95%:'); print(ci_f.round(4).to_string(index=False)); print(diff_f.round(4).to_string(index=False))
    return tab, ci_m, diff_m, ci_f, diff_f

T2 = tier2_table('holdout', 'Tầng 2 — Hold-out, gold-actionable')

Tầng 2 — Hold-out, gold-actionable (n = 183; %):
        chiến lược   Macro-F1     F1 BUG F1 FEATURE      F1 UX  số lần chạy
          ViSoBERT 76.5 ± 1.6 85.6 ± 0.9 57.8 ± 3.3 86.0 ± 0.6            3
Bi-LSTM + fastText 69.6 ± 1.1 79.6 ± 0.5 50.5 ± 3.5 78.8 ± 1.5            3
      LLM few-shot       89.7       94.0       83.3       91.9            1

Macro-F1 — CI 95%:
          cấu hình  giá trị  CI 2,5%  CI 97,5%
          ViSoBERT   0.7719   0.6954    0.8383
Bi-LSTM + fastText   0.7011   0.6164    0.7764
      LLM few-shot   0.8972   0.8383    0.9454
                 A                  B  chênh (A − B)  CI 2,5%  CI 97,5%  CI chứa 0
          ViSoBERT Bi-LSTM + fastText         0.0708  -0.0085    0.1526       True
          ViSoBERT       LLM few-shot        -0.1253  -0.1947   -0.0608      False
Bi-LSTM + fastText       LLM few-shot        -0.1961  -0.2872   -0.1078      False

F1 FEATURE — CI 95%:
          cấu hình  giá trị  CI 2,5%  CI 97,5%
          ViSoBERT   0.5882   0.4082  

## End-to-end bốn lớp trên Hold-out: cascade và baseline phẳng

**Cách ghép**

Cascade của mỗi chiến lược: tầng 1 gọi NOT_ACTIONABLE thì nhãn OTHER, ngược lại lấy nhãn tầng 2 của cùng chiến lược và cùng seed.
Baseline phẳng là ViSoBERT bốn lớp.

**Chỉ số**

Macro-F1 bốn lớp, F1 từng lớp, khoảng tin cậy, chênh lệch ghép cặp và ma trận nhầm lẫn (seed 42 / LLM).

In [5]:
def cascade_runs(s):
    A, T = tier1_act(s), tier2_ids(s)
    out = {}
    for name in STRATS:
        if name not in A:
            continue
        runs = {}
        for key, act in A[name].items():
            t2 = T[name][key]
            assert not ((t2 < 0) & act).any(), f'{s}/{name}: tầng 2 thiếu dự đoán cho review tầng 1 giữ lại'
            runs[key] = E.cascade_labels(act, np.where(t2 < 0, 0, t2))
        out[f'Cascade {name}'] = runs
    return out

def e2e_table(s, runs_by_name, title):
    rows = []
    for name, runs in runs_by_name.items():
        ms = [E.metrics(Y4[s], p, FOUR) for p in runs.values()]
        rows.append({'cấu hình': name, 'Macro-F1 (4 lớp)': pm([m['macro_f1'] for m in ms]),
                     **{f'F1 {c}': pm([m[f'f1_{c}'] for m in ms]) for c in FOUR}, 'số lần chạy': len(runs)})
    tab = pd.DataFrame(rows)
    print(f'{title} (n = {len(Y4[s])}; %):'); print(tab.to_string(index=False))
    ci, diff = bootstrap_table(Y4[s], {n: canon(r) for n, r in runs_by_name.items()}, 4)
    print('\nMacro-F1 bốn lớp — CI 95%:'); print(ci.round(4).to_string(index=False)); print(diff.round(4).to_string(index=False))
    return tab, ci, diff

E2E_RUNS = {**cascade_runs('holdout'), 'Baseline phẳng ViSoBERT': flat_ids('holdout')}
E2E = e2e_table('holdout', E2E_RUNS, 'End-to-end — Hold-out')
CONF = {}
for name, runs in E2E_RUNS.items():
    CONF[name] = E.confusion(Y4['holdout'], canon(runs), FOUR)
    print(f'\nMa trận nhầm lẫn — {name} (seed 42 / LLM):'); print(CONF[name].to_string())

End-to-end — Hold-out (n = 1000; %):
                  cấu hình Macro-F1 (4 lớp)     F1 BUG F1 FEATURE      F1 UX   F1 OTHER  số lần chạy
          Cascade ViSoBERT       62.1 ± 0.7 48.5 ± 2.0 34.8 ± 3.6 73.2 ± 0.4 91.8 ± 0.5            3
Cascade Bi-LSTM + fastText       53.5 ± 3.7 40.2 ± 2.1 22.2 ± 7.3 66.1 ± 3.6 85.5 ± 5.2            3
      Cascade LLM few-shot             81.3       71.9       69.1       88.5       95.6            1
   Baseline phẳng ViSoBERT       66.8 ± 1.5 47.0 ± 1.4 46.1 ± 3.6 79.7 ± 0.8 94.4 ± 0.3            3

Macro-F1 bốn lớp — CI 95%:
                  cấu hình  giá trị  CI 2,5%  CI 97,5%
          Cascade ViSoBERT   0.6290   0.5759    0.6776
Cascade Bi-LSTM + fastText   0.5419   0.4989    0.5829
      Cascade LLM few-shot   0.8129   0.7632    0.8554
   Baseline phẳng ViSoBERT   0.6670   0.6034    0.7209
                         A                          B  chênh (A − B)  CI 2,5%  CI 97,5%  CI chứa 0
          Cascade ViSoBERT Cascade Bi-LSTM + fastText   

## Focused FEATURE/UX set (báo cáo tách biệt)

**Tính chất của tập**

Focused set gồm 150 review được chọn theo cụm từ gợi ý FEATURE/UX, nên là tập làm giàu, không phản ánh phân bố tự nhiên. Tập chỉ dùng
để đo khả năng phân biệt lớp hiếm ở tầng 2; kết quả không gộp vào chỉ số chính.

In [6]:
FOC = tier2_table('focused', 'Tầng 2 — focused set (tập làm giàu), gold-actionable')

Tầng 2 — focused set (tập làm giàu), gold-actionable (n = 91; %):
        chiến lược   Macro-F1      F1 BUG F1 FEATURE      F1 UX  số lần chạy
          ViSoBERT 76.2 ± 3.2 57.7 ± 11.0 91.0 ± 1.5 80.0 ± 0.0            3
Bi-LSTM + fastText 68.8 ± 3.5 42.9 ± 10.2 85.4 ± 1.4 78.1 ± 1.8            3
      LLM few-shot       75.0        55.6       91.4       78.0            1

Macro-F1 — CI 95%:
          cấu hình  giá trị  CI 2,5%  CI 97,5%
          ViSoBERT   0.7898   0.6746    0.8829
Bi-LSTM + fastText   0.7261   0.6025    0.8294
      LLM few-shot   0.7498   0.6179    0.8587
                 A                  B  chênh (A − B)  CI 2,5%  CI 97,5%  CI chứa 0
          ViSoBERT Bi-LSTM + fastText         0.0637  -0.0530    0.1850       True
          ViSoBERT       LLM few-shot         0.0399  -0.1021    0.1831       True
Bi-LSTM + fastText       LLM few-shot        -0.0237  -0.1925    0.1427       True

F1 FEATURE — CI 95%:
          cấu hình  giá trị  CI 2,5%  CI 97,5%
          ViSoBER

## External Test Lazada

**Mục đích**

Kiểm tra khả năng khái quát sang ứng dụng khác bằng cascade end-to-end của ba chiến lược, với mô hình và prompt đã đóng băng, không
tinh chỉnh lại. Lazada có phân bố khác Shopee (1★ khoảng 60%).

In [7]:
LAZ = e2e_table('lazada', cascade_runs('lazada'), 'End-to-end — Lazada')

End-to-end — Lazada (n = 500; %):
                  cấu hình Macro-F1 (4 lớp)     F1 BUG F1 FEATURE      F1 UX   F1 OTHER  số lần chạy
          Cascade ViSoBERT       56.0 ± 0.4 42.4 ± 1.6 33.8 ± 2.1 59.0 ± 2.4 88.8 ± 1.3            3
Cascade Bi-LSTM + fastText       47.1 ± 6.4 31.9 ± 5.5 21.3 ± 6.7 55.3 ± 5.9 80.0 ± 8.3            3
      Cascade LLM few-shot             60.5       48.1       37.5       66.7       89.8            1

Macro-F1 bốn lớp — CI 95%:
                  cấu hình  giá trị  CI 2,5%  CI 97,5%
          Cascade ViSoBERT   0.5591   0.4864    0.6254
Cascade Bi-LSTM + fastText   0.4925   0.4303    0.5506
      Cascade LLM few-shot   0.6050   0.5214    0.6774
                         A                          B  chênh (A − B)  CI 2,5%  CI 97,5%  CI chứa 0
          Cascade ViSoBERT Cascade Bi-LSTM + fastText         0.0666   0.0126    0.1197      False
          Cascade ViSoBERT       Cascade LLM few-shot        -0.0458  -0.1370    0.0453       True
Cascade Bi-LSTM +

## Hybrid 3×3 trên Hold-out

**Cách ghép**

Tầng 1 của chiến lược X ghép với tầng 2 của chiến lược Y (seed 42 / LLM), từ dự đoán đã lưu, không train lại. Đường chéo là ba cascade
thuần.

**Chi phí của mỗi tổ hợp**

Thời gian và chi phí API cho 1.000 review = chi phí tầng 1 trên toàn bộ review + chi phí tầng 2 trên phần tầng 1 giữ lại (tỷ lệ giữ
lại đo trên Hold-out).

In [8]:
A, T = tier1_act('holdout'), tier2_ids('holdout')
HY = hybrid_predictions({x: canon(A[x]) for x in STRATS}, {y: canon(T[y]) for y in STRATS})
HY_CI, _ = bootstrap_table(Y4['holdout'], {f'{x} × {y}': p for (x, y), p in HY.items()}, 4)
CN = pd.read_csv(os.path.join(RES, 'gd10_costs_neural.csv')).set_index(['tier', 'model'])
CL = pd.read_csv(os.path.join(RES, 'gd10_llm_costs.csv')).set_index('bước')
SEC = {1: {n: CN.loc[('tier1', m), 'infer_seconds_per_1000_mean'] for n, m in NEURAL.items()},
       2: {n: CN.loc[('tier2', m), 'infer_seconds_per_1000_mean'] for n, m in NEURAL.items()}}
USD = {1: {n: 0.0 for n in NEURAL}, 2: {n: 0.0 for n in NEURAL}}
SEC[1]['LLM few-shot'], USD[1]['LLM few-shot'] = CL.loc['holdout – tầng 1', 'giây_/1000_review'], CL.loc['holdout – tầng 1', 'usd_/1000_review']
SEC[2]['LLM few-shot'], USD[2]['LLM few-shot'] = CL.loc['holdout – tầng 2', 'giây_/1000_review'], CL.loc['holdout – tầng 2', 'usd_/1000_review']
rows = []
for (x, y), p in HY.items():
    keep = float(canon(A[x]).mean())
    ci = HY_CI.set_index('cấu hình').loc[f'{x} × {y}']
    rows.append({'tầng 1 (X)': x, 'tầng 2 (Y)': y, 'thuần': x == y, 'Macro-F1 (4 lớp)': round(ci['giá trị'] * 100, 1),
                 'CI 95%': f"{ci['CI 2,5%'] * 100:.1f}–{ci['CI 97,5%'] * 100:.1f}", 'tỷ lệ tầng 1 giữ lại': round(keep, 3),
                 'giây / 1.000 review': round(SEC[1][x] + keep * SEC[2][y], 2),
                 'USD API / 1.000 review': round(USD[1][x] + keep * USD[2][y], 4)})
HYB = pd.DataFrame(rows)
print('Hybrid 3×3 — Hold-out (seed 42 / LLM):'); print(HYB.to_string(index=False))
print(); print(HYB.pivot(index='tầng 1 (X)', columns='tầng 2 (Y)', values='Macro-F1 (4 lớp)').loc[STRATS, STRATS].to_string())

Hybrid 3×3 — Hold-out (seed 42 / LLM):
        tầng 1 (X)         tầng 2 (Y)  thuần  Macro-F1 (4 lớp)    CI 95%  tỷ lệ tầng 1 giữ lại  giây / 1.000 review  USD API / 1.000 review
          ViSoBERT           ViSoBERT   True              62.9 57.6–67.8                 0.199                 1.55                  0.0000
          ViSoBERT Bi-LSTM + fastText  False              58.9 53.6–63.7                 0.199                 1.28                  0.0000
          ViSoBERT       LLM few-shot  False              69.5 63.9–74.4                 0.199               185.17                  0.0320
Bi-LSTM + fastText           ViSoBERT  False              58.9 54.2–63.4                 0.306                 0.51                  0.0000
Bi-LSTM + fastText Bi-LSTM + fastText   True              54.2 49.9–58.3                 0.306                 0.11                  0.0000
Bi-LSTM + fastText       LLM few-shot  False              65.6 60.3–70.2                 0.306               282.87      

## Chi phí tổng hợp (cost-aware)

**Các chiều chi phí**

Nhãn cần để xây dựng, thời gian huấn luyện, thời gian suy luận cho 1.000 review (cascade = tầng 1 + tầng 2 trên phần giữ lại), bộ nhớ
GPU, chi phí API thực và mức phụ thuộc nhà cung cấp. Thời gian gán nhãn thực tế chưa được ghi nhận và để trống.

In [9]:
keep = {n: float(canon(A[n]).mean()) for n in STRATS}
COST = []
for n, m in NEURAL.items():
    COST.append({'chiến lược': n, 'nhãn cần': 'Train 1.200 + Validation 350',
                 'huấn luyện / seed (giây, tầng 1 + tầng 2)': round(CN.loc[('tier1', m), 'train_seconds_mean'] + CN.loc[('tier2', m), 'train_seconds_mean'], 1),
                 'suy luận / 1.000 review (giây)': round(SEC[1][n] + keep[n] * SEC[2][n], 2),
                 'GPU tối đa (GB)': max(CN.loc[('tier1', m), 'peak_gpu_gb_max'], CN.loc[('tier2', m), 'peak_gpu_gb_max']),
                 'USD API / 1.000 review': 0.0, 'phụ thuộc nhà cung cấp': 'Không (chạy cục bộ)'})
COST.append({'chiến lược': 'LLM few-shot', 'nhãn cần': '17 ví dụ (Train) + Validation 350', 'huấn luyện / seed (giây, tầng 1 + tầng 2)': 0.0,
             'suy luận / 1.000 review (giây)': round(SEC[1]['LLM few-shot'] + keep['LLM few-shot'] * SEC[2]['LLM few-shot'], 1),
             'GPU tối đa (GB)': 0.0, 'USD API / 1.000 review': round(USD[1]['LLM few-shot'] + keep['LLM few-shot'] * USD[2]['LLM few-shot'], 4),
             'phụ thuộc nhà cung cấp': 'Có (OpenAI, snapshot cố định)'})
COST.append({'chiến lược': 'Baseline phẳng ViSoBERT', 'nhãn cần': 'Train 1.200 + Validation 350',
             'huấn luyện / seed (giây, tầng 1 + tầng 2)': round(CN.loc[('flat', 'visobert'), 'train_seconds_mean'], 1),
             'suy luận / 1.000 review (giây)': round(CN.loc[('flat', 'visobert'), 'infer_seconds_per_1000_mean'], 2),
             'GPU tối đa (GB)': CN.loc[('flat', 'visobert'), 'peak_gpu_gb_max'], 'USD API / 1.000 review': 0.0,
             'phụ thuộc nhà cung cấp': 'Không (chạy cục bộ)'})
COST = pd.DataFrame(COST)
COST['giờ gán nhãn'] = '{...}'
print(COST.to_string(index=False))
print('GPU khi train lại:', sorted(set(CN['gpu'])))

             chiến lược                          nhãn cần  huấn luyện / seed (giây, tầng 1 + tầng 2)  suy luận / 1.000 review (giây)  GPU tối đa (GB)  USD API / 1.000 review        phụ thuộc nhà cung cấp giờ gán nhãn
               ViSoBERT      Train 1.200 + Validation 350                                       42.7                            1.55            2.509                  0.0000           Không (chạy cục bộ)        {...}
     Bi-LSTM + fastText      Train 1.200 + Validation 350                                        6.4                            0.11            0.624                  0.0000           Không (chạy cục bộ)        {...}
           LLM few-shot 17 ví dụ (Train) + Validation 350                                        0.0                         1103.50            0.000                  0.2064 Có (OpenAI, snapshot cố định)        {...}
Baseline phẳng ViSoBERT      Train 1.200 + Validation 350                                       39.7                            1.39

## Lưu kết quả

**Các file**

Mọi bảng được lưu với tiền tố `gd10_` trong `05_experiments/rq1_classification/`. Ngày mở và commit in ở đầu notebook dùng để cập nhật
`01_data/HOLDOUT_LOCK.md`.

In [10]:
def save(df, name):
    df.to_csv(os.path.join(RES, f'gd10_{name}.csv'), index=False)

save(T1, 'tier1_holdout'); save(pd.concat([c for c, _ in T1_CI]), 'tier1_holdout_ci'); save(pd.concat([d for _, d in T1_CI]), 'tier1_holdout_diff')
save(T2[0], 'tier2_holdout'); save(T2[1], 'tier2_holdout_ci'); save(T2[2], 'tier2_holdout_diff'); save(T2[3], 'tier2_holdout_feature_ci')
save(E2E[0], 'e2e_holdout'); save(E2E[1], 'e2e_holdout_ci'); save(E2E[2], 'e2e_holdout_diff')
pd.concat([c.assign(cấu_hình=n) for n, c in CONF.items()]).to_csv(os.path.join(RES, 'gd10_confusion_holdout.csv'))
save(FOC[0], 'tier2_focused'); save(FOC[1], 'tier2_focused_ci'); save(FOC[3], 'tier2_focused_feature_ci')
save(LAZ[0], 'e2e_lazada'); save(LAZ[1], 'e2e_lazada_ci'); save(LAZ[2], 'e2e_lazada_diff')
save(HYB, 'hybrid_holdout'); save(COST, 'cost_summary')
json.dump({'open_date': OPEN_DATE, 'commit': COMMIT, 'uncommitted_changes': DIRTY}, open(os.path.join(RES, 'gd10_open_info.json'), 'w'), indent=1)
print('Đã lưu:', sorted(f for f in os.listdir(RES) if f.startswith('gd10_')))

Đã lưu: ['gd10_confusion_holdout.csv', 'gd10_cost_summary.csv', 'gd10_costs_neural.csv', 'gd10_e2e_holdout.csv', 'gd10_e2e_holdout_ci.csv', 'gd10_e2e_holdout_diff.csv', 'gd10_e2e_lazada.csv', 'gd10_e2e_lazada_ci.csv', 'gd10_e2e_lazada_diff.csv', 'gd10_gate_recall.csv', 'gd10_gate_validation.csv', 'gd10_hybrid_holdout.csv', 'gd10_llm_calls.jsonl', 'gd10_llm_costs.csv', 'gd10_llm_focused_tier2.csv', 'gd10_llm_holdout_tier1.csv', 'gd10_llm_holdout_tier2.csv', 'gd10_llm_lazada_tier1.csv', 'gd10_llm_lazada_tier2.csv', 'gd10_llm_pilot_calls.jsonl', 'gd10_open_info.json', 'gd10_pred_focused.csv', 'gd10_pred_holdout.csv', 'gd10_pred_lazada.csv', 'gd10_pred_validation.csv', 'gd10_run_info.json', 'gd10_tier1_holdout.csv', 'gd10_tier1_holdout_ci.csv', 'gd10_tier1_holdout_diff.csv', 'gd10_tier2_focused.csv', 'gd10_tier2_focused_ci.csv', 'gd10_tier2_focused_feature_ci.csv', 'gd10_tier2_holdout.csv', 'gd10_tier2_holdout_ci.csv', 'gd10_tier2_holdout_diff.csv', 'gd10_tier2_holdout_feature_ci.csv']


## Nhận xét

**Nguồn số liệu**

Mọi số dưới đây lấy từ các file `gd10_*` trong `05_experiments/rq1_classification/`. Mô hình neural được báo bằng trung bình ± độ lệch
chuẩn qua ba seed; khoảng tin cậy, chênh lệch ghép cặp, ma trận nhầm lẫn và hybrid 3×3 dùng seed 42. LLM là một lần chạy.

**End-to-end bốn lớp trên Hold-out**

Macro-F1 bốn lớp: cascade LLM few-shot 81,3 (CI 76,3–85,5), baseline phẳng ViSoBERT 66,8 ± 1,5, cascade ViSoBERT 62,1 ± 0,7, cascade
Bi-LSTM 53,5 ± 3,7. CI của LLM không chồng lấn CI seed 42 của ba cấu hình còn lại (baseline 60,3–72,1; cascade ViSoBERT 57,6–67,8;
cascade Bi-LSTM 49,9–58,3). CI của chênh lệch ghép cặp cũng không chứa 0 ở cả ba cặp: LLM hơn baseline 14,6 điểm (CI 8,6–21,0), hơn
cascade ViSoBERT 18,4 điểm (CI 13,1–23,7) và hơn cascade Bi-LSTM 27,1 điểm (CI 21,9–32,4). Theo quy ước đã khóa, đây là các chênh
lệch có ý nghĩa thống kê.

**Cascade neural so với baseline phẳng**

Cả hai cascade neural có Macro-F1 thấp hơn baseline phẳng. Với Bi-LSTM, chênh −12,5 điểm (CI −18,2 đến −6,7) có ý nghĩa thống kê. Với
ViSoBERT, chênh ở seed 42 là −3,8 điểm và CI (−8,6 đến +1,2) chứa 0; theo quy ước đã khóa, cascade ViSoBERT và baseline phẳng được
xem là tương đương trên Hold-out, và mức thấp hơn 4,7 điểm của trung bình ba seed (62,1 so với 66,8) là xu hướng, chưa phải khác biệt
có ý nghĩa. Trên Validation hai cấu hình gần như bằng nhau (68,8 ± 0,8 và 69,1 ± 2,0).

**Nhiễu tầng 1 lan truyền xuống tầng 2**

Ma trận nhầm lẫn seed 42 cho thấy cascade ViSoBERT đưa 75 trên 817 review OTHER sang lớp actionable (32 thành BUG, 24 thành FEATURE, 19
thành UX), trong khi baseline phẳng chỉ đưa 24 review (13, 7, 4). Ảnh hưởng rõ nhất ở FEATURE, lớp chỉ có 24 review trong Hold-out:
cascade đoán FEATURE 43 lần, đúng 13 lần, và 24 lượt đến từ review OTHER; baseline đoán FEATURE 20 lần, đúng 10 lần, 7 lượt đến từ
OTHER. Cascade tìm được nhiều review FEATURE hơn nhưng precision thấp hơn nhiều, nên F1 FEATURE end-to-end thấp hơn: 34,8 ± 3,6 so với
46,1 ± 3,6. Ở BUG, cả hai cấu hình đều bỏ sót nhiều: 45 (cascade) và 51 (baseline) trên 89 review BUG bị đoán là OTHER.

**Tầng 1 trên Hold-out**

Precision / recall ACTIONABLE: LLM 77,2 / 86,9; ViSoBERT 62,7 ± 2,9 / 69,8 ± 4,0; Bi-LSTM 47,0 ± 7,5 / 84,0 ± 7,4. F1 ACTIONABLE của
LLM (81,7) cao hơn ViSoBERT (65,9 ± 0,9) và Bi-LSTM (59,7 ± 4,7), với CI chênh lệch không chứa 0. Recall của LLM và Bi-LSTM tương đương
(CI chênh lệch −10,2 đến +2,5 chứa 0), còn precision của LLM cao hơn 30 điểm: ở mức recall tương đương, LLM đưa ít review OTHER sang
tầng 2 hơn hẳn (47 trên 817 so với 154 trên 817 của Bi-LSTM, seed 42).

**Recall ViSoBERT dưới sàn 0,80**

Ngưỡng 0,11 của ViSoBERT cho recall 0,817 trên Validation ở seed 42, nhưng trên Hold-out recall chỉ còn 69,8 ± 4,0 (seed 42: 67,8, CI
61,0–74,4), dưới sàn 0,80 của protocol. Bi-LSTM ở ngưỡng 0,09 giữ được 84,0 ± 7,4 (seed 42: 83,1). Ngưỡng là quy tắc đã khóa nên không
được chỉnh lại sau khi mở Hold-out. Đây là giới hạn được ghi nhận: ngưỡng chọn trên 71 review actionable của Validation không giữ
nguyên mức recall khi chuyển sang dữ liệu mới.

**Tầng 2 trên gold-actionable của Hold-out**

Macro-F1 ba lớp: LLM 89,7, ViSoBERT 76,5 ± 1,6, Bi-LSTM 69,6 ± 1,1. LLM cao hơn cả hai mô hình neural (CI chênh lệch không chứa 0);
ViSoBERT và Bi-LSTM tương đương (CI chứa 0). F1 FEATURE: LLM 83,3, ViSoBERT 57,8 ± 3,3, Bi-LSTM 50,5 ± 3,5. Với 24 review FEATURE, CI
rất rộng (ViSoBERT 40,8–73,5) nhưng chênh lệch với LLM vẫn không chứa 0. Như vậy, trên phân bố tự nhiên, phần hơn của LLM ở end-to-end
không chỉ đến từ tầng 1 mà còn từ tầng 2.

**Focused set (tập làm giàu)**

Focused set được chọn theo các cụm từ gợi ý FEATURE/UX: 91 review gold-actionable, trong đó 56 review FEATURE. Tập này không phản ánh
phân bố tự nhiên và không được gộp vào chỉ số chính. F1 FEATURE: LLM 91,4, ViSoBERT 91,0 ± 1,5, Bi-LSTM 85,4 ± 1,4; mọi chênh lệch ghép
cặp đều chứa 0. Macro-F1 cũng không khác biệt có ý nghĩa (LLM 75,0; ViSoBERT 76,2 ± 3,2; Bi-LSTM 68,8 ± 3,5). Khi FEATURE đủ ca và là
lớp đa số, ba chiến lược nhận ra FEATURE ở mức ngang nhau. Trên Hold-out, FEATURE chỉ chiếm 24 trên 183 review actionable và F1 FEATURE
của tầng 2 neural thấp hơn LLM rõ rệt. Hai kết quả cho thấy năng lực FEATURE của tầng 2 neural phụ thuộc mạnh vào phân bố lớp; focused
set không đủ để kết luận tầng 2 neural không phải điểm yếu. Hai tập khác phân bố nên các con số không so trực tiếp với nhau.

**Hybrid 3×3: tách đóng góp của hai tầng**

Xuất phát từ cascade ViSoBERT thuần (seed 42: 62,9, CI 57,6–67,8): thay tầng 1 bằng LLM đạt 73,3 (CI 67,8–78,3), tăng 10,4 điểm; thay
tầng 2 bằng LLM đạt 69,5 (CI 63,9–74,4), tăng 6,6 điểm; thay cả hai tầng là cascade LLM 81,3, tăng 18,4 điểm. Xuất phát từ cascade
Bi-LSTM thuần (54,2): thay tầng 1 bằng LLM tăng 15,1 điểm (69,3), thay tầng 2 bằng LLM tăng 11,4 điểm (65,6). Ở cả hai mô hình, đổi tầng
1 mang lại phần tăng lớn hơn đổi tầng 2, nên tầng 1 là nút thắt lớn hơn của cascade neural. Tầng 2 vẫn đóng góp đáng kể: với cùng tầng 1
LLM, tầng 2 LLM cao hơn tầng 2 ViSoBERT 8,0 điểm (81,3 so với 73,3). Hybrid chỉ có CI của từng tổ hợp, không có kiểm định chênh lệch
ghép cặp riêng; giá trị 62,9 là seed 42, khác trung bình ba seed 62,1 ở bảng end-to-end.

**External Test Lazada**

Macro-F1 bốn lớp: cascade LLM 60,5 (CI 52,1–67,7), cascade ViSoBERT 56,0 ± 0,4, cascade Bi-LSTM 47,1 ± 6,4. LLM hơn ViSoBERT 4,6 điểm
nhưng CI (−4,5 đến +13,7) chứa 0, nên hai cấu hình tương đương trên Lazada; LLM hơn Bi-LSTM 11,2 điểm (CI 3,0–19,5). Cả ba cấu hình đều
giảm so với Hold-out Shopee (81,3; 62,1; 53,5), và khoảng cách giữa LLM và ViSoBERT hẹp lại khi chuyển sang ứng dụng khác.

**Chi phí và mức tự chủ**

- LLM few-shot: 17 ví dụ lấy từ Train, không huấn luyện; prompt v1 được đánh giá trên Validation 350 có nhãn trước khi khóa. Suy luận
  1.103,5 giây cho 1.000 review (gọi API tuần tự, mỗi lượt một review), chi phí $0,2064 cho 1.000 review, phụ thuộc OpenAI với
  snapshot `gpt-4o-mini-2024-07-18`. Toàn bộ lần chạy GĐ10 gồm 2.195 lượt gọi, 0 lần thử lại do parse, 0 fallback, tổng $0,373.
- ViSoBERT: huấn luyện 42,7 giây mỗi seed (tầng 1 + tầng 2, Tesla T4), suy luận 1,55 giây cho 1.000 review, GPU tối đa 2,51 GB, chạy cục
  bộ.
- Bi-LSTM + fastText: huấn luyện 6,4 giây, suy luận 0,11 giây cho 1.000 review, GPU 0,62 GB, chạy cục bộ; hiệu năng thấp nhất ở mọi
  bảng Hold-out.
- Hai chiến lược neural cần Train 1.200 và Validation 350 có nhãn. Thời gian gán nhãn thực tế chưa được ghi nhận.
- Thời gian suy luận của LLM gấp khoảng 700 lần ViSoBERT và khoảng 10.000 lần Bi-LSTM.

**Đánh đổi, không xếp hạng tuyệt đối**

Trên Hold-out Shopee, cascade LLM có hiệu năng cao nhất với khác biệt có ý nghĩa thống kê. Đổi lại, LLM có thời gian suy luận lớn nhất,
chi phí API tính theo lượt gọi và phụ thuộc nhà cung cấp bên ngoài (review phải gửi ra ngoài; snapshot có thể bị ngừng). ViSoBERT chạy
cục bộ, chi phí suy luận thấp, hiệu năng ở giữa và tương đương LLM trên Lazada. Bi-LSTM rẻ và nhanh nhất nhưng hiệu năng thấp nhất. Các
tổ hợp hybrid là điểm trung gian: tầng 1 ViSoBERT + tầng 2 LLM đạt 69,5 với 185 giây và $0,032 cho 1.000 review; tầng 1 LLM + tầng 2
ViSoBERT đạt 73,3 với 913 giây và $0,173. Lựa chọn phụ thuộc ràng buộc của BA: ngân sách API, yêu cầu dữ liệu không rời hệ thống,
khối lượng review và độ trễ chấp nhận được. Kết quả LLM là một lần chạy với một snapshot và một bộ ví dụ chọn ngẫu nhiên (giới hạn đã
ghi ở Giai đoạn 8), nên không có độ dao động giữa các lần chạy như ba seed của neural.

## So sánh Validation và Hold-out

**Bảng đối chiếu**

| chỉ số | chiến lược | Validation | Hold-out |
|---|---|---|---|
| F1 ACTIONABLE | ViSoBERT | 76,5 ± 0,6 | 65,9 ± 0,9 |
| F1 ACTIONABLE | Bi-LSTM + fastText | 68,5 ± 4,2 | 59,7 ± 4,7 |
| F1 ACTIONABLE | LLM few-shot | 82,7 | 81,7 |
| Recall ACTIONABLE | ViSoBERT | 82,6 ± 4,3 | 69,8 ± 4,0 |
| Recall ACTIONABLE | Bi-LSTM + fastText | 87,8 ± 6,4 | 84,0 ± 7,4 |
| Recall ACTIONABLE | LLM few-shot | 87,3 | 86,9 |
| Macro-F1 tầng 2 | ViSoBERT | 75,6 ± 2,2 | 76,5 ± 1,6 |
| Macro-F1 tầng 2 | Bi-LSTM + fastText | 72,0 ± 3,2 | 69,6 ± 1,1 |
| Macro-F1 tầng 2 | LLM few-shot | 79,9 | 89,7 |
| Macro-F1 end-to-end | Cascade ViSoBERT | 68,8 ± 0,8 | 62,1 ± 0,7 |
| Macro-F1 end-to-end | Baseline phẳng ViSoBERT | 69,1 ± 2,0 | 66,8 ± 1,5 |

**Tầng 1 neural giảm, LLM gần như giữ nguyên**

F1 ACTIONABLE giảm 10,6 điểm với ViSoBERT và 8,8 điểm với Bi-LSTM, còn LLM chỉ giảm 1,0 điểm. Cấu hình và ngưỡng tầng 1 của hai mô hình
neural được chọn trên chính Validation; prompt LLM v1 được đánh giá một lần trên Validation và giữ nguyên, không có vòng tối ưu nào trên
tập này. Tỷ lệ review actionable của hai tập gần nhau (71 trên 350, tức 20,3%, so với 183 trên 1.000, tức 18,3%).

**Tầng 2 chịu ảnh hưởng của phân bố lớp**

Validation đã được cân lại ở amendment v1.8 (22 FEATURE trên 71 review actionable), còn Hold-out giữ phân bố tự nhiên (24 FEATURE trên
183). Thay đổi ở tầng 2, kể cả mức tăng của LLM, vì vậy phản ánh cả khác biệt phân bố lớp chứ không chỉ độ lạc quan của Validation.

**Cascade so với baseline đổi chiều**

Trên Validation cascade ViSoBERT và baseline phẳng gần như hòa (−0,3 điểm). Trên Hold-out cascade thấp hơn 4,7 điểm theo trung bình
ba seed (seed 42: −3,8, CI chứa 0), còn LLM hơn baseline rõ hơn.

**Số dùng để kết luận**

Kết luận RQ1 dùng số Hold-out. Số Validation chỉ phục vụ việc chọn cấu hình, ngưỡng và kiểm tra cổng Bước 0, và lạc quan vì được tune
trên chính tập đó.